# Exercise 3 - Transition Matrices and Markov Chains

This notebook solves Parts A-D of Exercise 3. The goal is not only to get the numbers, but also to connect the algebra to the Python workflow.

A transition matrix acts on a column vector `P(t)`. In Python we write this multiplication as `T @ P`, and after `t` steps we use `np.linalg.matrix_power(T, t) @ P0`.

In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("Matplotlib is not installed in this Python environment. The numerical cells will still run; use Colab or install matplotlib to render the plot.")

np.set_printoptions(precision=6, suppress=True)

P0 = np.array([0, 0, 1, 0, 0], dtype=float)

T_A = np.array([
    [1/2, 1/2, 0,   0,   0],
    [1/2, 0,   1/2, 0,   0],
    [0,   1/2, 0,   1/2, 0],
    [0,   0,   1/2, 0,   1/2],
    [0,   0,   0,   1/2, 1/2],
], dtype=float)

T_B = np.array([
    [0, 1/2, 0,   0,   0],
    [1, 0,   1/2, 0,   0],
    [0, 1/2, 0,   1/2, 0],
    [0, 0,   1/2, 0,   1],
    [0, 0,   0,   1/2, 0],
], dtype=float)

def state_after(T, t, P=P0):
    """Return P(t) = T^t P(0)."""
    return np.linalg.matrix_power(T, t) @ P

def print_vector(label, vector):
    print(f"{label} = {vector}")

## Part A

The first matrix is column-stochastic: every column sums to 1. That is the Markov-chain version of conservation of total probability.

The general formula is:

`P(t) = T^t P(0)`

In Python, the matching expression is:

`np.linalg.matrix_power(T, t) @ P0`

In [ ]:
print("Column sums for Part A:", T_A.sum(axis=0))
print_vector("P_A(2)", state_after(T_A, 2))
print_vector("P_A(3)", state_after(T_A, 3))

eigvals_A, eigvecs_A = np.linalg.eig(T_A)
order_A = np.argsort(-eigvals_A.real)
eigvals_A = eigvals_A[order_A]
eigvecs_A = eigvecs_A[:, order_A]

print("\nEigenvalues for Part A:")
print(eigvals_A.real)
print("\nEigenvectors are columns of this matrix. They may differ by sign or scale from hand-written eigenvectors:")
print(eigvecs_A.real)

det_A = np.linalg.det(T_A)
print("\ndet(T_A) =", det_A)
print("\nInverse of T_A:")
print(np.linalg.inv(T_A))

print("\nLarge-time checks:")
for t in [20, 50, 100]:
    print_vector(f"P_A({t})", state_after(T_A, t))

### Part A interpretation

- `P(2) = [1/4, 0, 1/2, 0, 1/4]^T`.
- `P(3) = [1/8, 3/8, 0, 3/8, 1/8]^T`.
- The eigenvalues are `cos(k*pi/5)` for `k = 0,1,2,3,4`, namely about `1, 0.809017, 0.309017, -0.309017, -0.809017`.
- `det(T_A) = 1/16`, so `T_A` is invertible.
- If `P(105)` is known exactly, then `P(104) = T_A^-1 P(105)`.
- For very large `t`, all eigenvalue components with absolute value less than 1 shrink, so a probability vector approaches `[1/5, 1/5, 1/5, 1/5, 1/5]^T`.

The subtle point: there is no real contradiction between invertibility and long-term convergence. Each finite power is invertible, but the limiting approximation throws away small components. Numerically, recovering the past from a very late rounded vector is unstable.

## Part B

The second matrix is also column-stochastic, but it behaves differently because it has eigenvalue `-1`. That causes an alternating long-term pattern.

In [ ]:
print("Column sums for Part B:", T_B.sum(axis=0))
print_vector("P_B(2)", state_after(T_B, 2))
print_vector("P_B(3)", state_after(T_B, 3))

eigvals_B, eigvecs_B = np.linalg.eig(T_B)
order_B = np.argsort(-eigvals_B.real)
eigvals_B = eigvals_B[order_B]
eigvecs_B = eigvecs_B[:, order_B]

print("\nEigenvalues for Part B:")
print(eigvals_B.real)
print("\nEigenvectors are columns of this matrix. They may differ by sign or scale from hand-written eigenvectors:")
print(eigvecs_B.real)

det_B = np.linalg.det(T_B)
print("\ndet(T_B) =", det_B)
print("Matrix rank =", np.linalg.matrix_rank(T_B))

print("\nLarge-time checks:")
for t in [20, 21, 100, 101]:
    print_vector(f"P_B({t})", state_after(T_B, t))

### Part B interpretation

- `P(2) = [1/4, 0, 1/2, 0, 1/4]^T`.
- `P(3) = [0, 1/2, 0, 1/2, 0]^T`.
- The eigenvalues are `1, 1/sqrt(2), 0, -1/sqrt(2), -1`.
- `det(T_B) = 0`, so `T_B` is not invertible.
- Since the matrix is singular, `P(104)` cannot generally be recovered uniquely from `P(105)`.
- For the given starting vector, the large-time behavior is a two-cycle:
  - even `t >= 2`: `[1/4, 0, 1/2, 0, 1/4]^T`
  - odd `t >= 3`: `[0, 1/2, 0, 1/2, 0]^T`

A different `P(0)` can change the long-term two-cycle because the final behavior depends on the components in the eigenvalue `1` and eigenvalue `-1` directions.

## Part C - A real-world story

A better story is a signal echoing along a one-dimensional line with five sample points.

- `P(t)` describes how much signal energy is present at each point after `t` time steps.
- In Part A, interior points split signal equally left and right. At an edge, half of the signal stays at the boundary and half moves inward. This acts like averaging, so the signal smooths out and converges toward an even spread.
- In Part B, the boundaries are perfect reflectors. Signal at an edge is sent fully back inward. For the given starting signal, the energy alternates between odd and even positions, so the process cycles instead of converging to one stable vector.

This story explains the important difference between the two matrices: Part A is a diffusion/averaging process, while Part B is an echo/reflection process with cyclic behavior.

## Part D - 100-state version

Now scale the Part A process to 100 states. Interior states send half their mass left and half right. The two edge states keep half their mass and send half inward.

For 100 states there are two central positions. We will use state 50 in one-indexed language, which is index 49 in Python.

In [ ]:
def reflecting_transition_matrix(n):
    T = np.zeros((n, n), dtype=float)
    for col in range(n):
        if col == 0:
            T[0, col] = 0.5
            T[1, col] = 0.5
        elif col == n - 1:
            T[n - 2, col] = 0.5
            T[n - 1, col] = 0.5
        else:
            T[col - 1, col] = 0.5
            T[col + 1, col] = 0.5
    return T

n = 100
T_100 = reflecting_transition_matrix(n)
P0_100 = np.zeros(n)
P0_100[49] = 1.0

times = [0, 50, 100, 500]
snapshots = {t: np.linalg.matrix_power(T_100, t) @ P0_100 for t in times}

print("Column sums are all 1:", np.allclose(T_100.sum(axis=0), 1))
for t in times:
    print(f"t={t:3d}: total={snapshots[t].sum():.6f}, peak={snapshots[t].max():.6f}")

if plt is None:
    print("Plot skipped because matplotlib is unavailable. In Colab, run this notebook as-is and the plot will render.")
else:
    states = np.arange(1, n + 1)
    plt.figure(figsize=(11, 5.5))
    for t in times:
        plt.plot(states, snapshots[t], label=f"t = {t}")

    plt.title("100-state reflecting signal model")
    plt.xlabel("State")
    plt.ylabel("Probability")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

### Part D interpretation

At `t = 0`, all signal mass is concentrated in the middle. As `t` increases, repeated multiplication by the transition matrix spreads the mass across neighboring states. The graph becomes wider and flatter because the signal is diffusing along the line. At very large times, the Part A-style reflecting signal process moves toward the uniform distribution.